[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)

# 01 · 锁定版 GPQA 评测 / Locked GPQA evaluation

所有排行榜分数都必须用这个 notebook（或等价的 `scripts/eval_gpqa.sh`）产出。**不要改评测参数。**

- 任务：`gpqa_diamond_cot_zeroshot`（主榜）+ `gpqa_main_cot_zeroshot`（第二指标）
- fp16，`max_model_len=8192`，`max_gen_toks=4096`，`temperature=0.6`，`top_p=0.95`，使用模型自带 chat template
- 4 个 vLLM 采样 seed 取平均，报告均值 ± 标准误
- 指标：`exact_match,flexible-extract`（取输出里最后一个 `(A)`–`(D)`）

⚠️ GPQA 条款要求不要在网上公开题目。`results/` 里的逐题日志包含题目，**只能留在你自己的 Colab / Drive 里**，只提交汇总后的 JSON。

时间：4B 模型每个 seed 大约需要（待 T4 实测）分钟。时间不够时先用 `SEEDS = "0"` 自测，正式提交再跑 4 个。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆比赛仓库（脚本都在里面） / clone the hackathon repo (branch: main)
import os
if not os.path.exists("GPQA-eval"):
    !git clone -q -b main https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q

In [ ]:
# 评测环境（vLLM + lm-eval）。约 3–5 分钟。装完如果提示重启 runtime，请重启后从下一格继续。
!pip install -q -r requirements-eval.txt
import importlib.metadata as md
for p in ["lm_eval", "vllm", "transformers", "torch"]:
    print(p, md.version(p if p != "lm_eval" else "lm-eval"))

In [ ]:
# GPQA 是 gated 数据集：先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款，
# 再用有 read（提交模型需要 write）权限的 token 登录。
# 推荐：Colab 左侧 🔑 Secrets 里添加 HF_TOKEN，这里就会自动读取。
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# ==== 只改这里 / edit only this cell ====
MODEL   = "Qwen/Qwen3-4B-Base"      # 你的公开 HF 模型仓库，例如 "my-team/qwen3-4b-sft"
TEAM    = "my-team"
BASE    = "Qwen/Qwen3-4B-Base"      # 或 "Qwen/Qwen3-1.7B-Base"
SEEDS   = "0 1 2 3"                   # 自测可先用 "0"
BACKEND = "vllm"                      # vLLM 跑不通时改 "hf"（慢很多）
# =======================================
OUT_DIR = f"results/{MODEL.replace('/', '__')}"

In [ ]:
!bash scripts/eval_gpqa.sh "$MODEL" "$OUT_DIR" "$SEEDS" "$BACKEND"

In [ ]:
# 汇总为提交 JSON，并写入实际用到的包版本
import json, importlib.metadata as md
sub_path = f"submissions/{TEAM}.json"
!python scripts/aggregate_results.py "$OUT_DIR" --model "$MODEL" --team "$TEAM" --base-model "$BASE" --backend "$BACKEND" --out "$sub_path"
sub = json.load(open(sub_path))
def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None
sub["eval"]["versions"] = {p: v for p in ["lm-eval", "vllm", "transformers", "torch"] if (v := _ver(p))}
json.dump(sub, open(sub_path, "w"), indent=2, ensure_ascii=False)
print(json.dumps(sub["scores"], indent=2))

## 提交

1. 打开 `submissions/<TEAM>.json`，把 `method`、`datasets`、`train_gpu_hours_t4`、`decontam`、`training_notebook` 里的 `TODO` 填完。
2. 下载这个 JSON（左侧文件栏右键 → Download）。
3. Fork https://github.com/yy726/GPQA-eval ，把它放进 `submissions/`，开 Pull Request。CI 会自动检查格式。
4. 助教会对前几名用同一配置复评，复评分数为准（`verified: true`）。

In [ ]:
# 可选：把逐题日志备份到你自己的 Google Drive（私有），方便之后做错误分析。不要上传到公开仓库。
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/gpqa-hackathon-results